# 04 — Avaliação e Exportação para o Power BI

Escolha do limiar de decisão e geração dos arquivos que alimentam o dashboard:
`predicoes.csv` e `metricas_modelo.csv`.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import ARQUIVO_BRUTO, ARQUIVO_TRATADO, COLUNA_ALVO

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [ ]:
import joblib
from sklearn.metrics import precision_recall_curve, ConfusionMatrixDisplay
from src.config import MODELS_DIR

X_teste, y_teste = joblib.load(MODELS_DIR / "conjunto_teste.joblib")
modelo = joblib.load(MODELS_DIR / "random_forest.joblib")
y_prob = modelo.predict_proba(X_teste)[:, 1]

## Curva Precision-Recall e escolha do limiar

In [ ]:
precisao, recall, limiares = precision_recall_curve(y_teste, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(recall, precisao)
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Curva Precision-Recall")
plt.grid(True)

In [ ]:
LIMIAR = 0.5  # ajustar com base na curva acima
y_prev = (y_prob >= LIMIAR).astype(int)
ConfusionMatrixDisplay.from_predictions(y_teste, y_prev)

## Exportar para o Power BI

In [ ]:
from src.models.predict_model import exportar_predicoes
from src.models.evaluate import main as exportar_metricas

exportar_predicoes("random_forest", X_teste, y_teste)
exportar_metricas()

Depois desta etapa, abrir o Power BI e atualizar as fontes em `data/processed/`.
Ver `powerbi/README.md`.